In [1]:
import os
import sys
import torch
import importlib
import pandas as pd
from sklearn.linear_model import Ridge

import utils
import evaluation_metrics as em
from datasets import load_dataset
from hkan.hkan import (
    Sigmoid, Gaussian, ReLU, Tanh, Softplus, Identity,
    make_hkan_layer, extend_hkan, set_tqdm_disable,
)
importlib.reload(utils)

/Users/gizemnurdal/miniconda3/envs/swim-meets-kans/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


<module 'utils' from '/Users/gizemnurdal/Workspace/swim-meets-kans/utils.py'>

# TF1 Experiments

## HKAN

In [2]:
# Load TF1 dataset
tf1 = load_dataset("TF1")
tf1_x_train, tf1_x_test, tf1_y_train, tf1_y_test = tf1["train_input"], tf1["test_input"], tf1["train_label"], tf1["test_label"]

In [3]:
print("TF1 train describe")
display(pd.DataFrame(tf1_x_train).describe())
print("TF1 test describe")
display(pd.DataFrame(tf1_x_test).describe())
print("TF1 y_train describe")
display(pd.Series(tf1_y_train).describe())
print("TF1 y_test describe")
display(pd.Series(tf1_y_test).describe())

TF1 train describe


,0,1
count,5000.000000,5000.000000
mean,0.502492,0.498871
std,0.291808,0.285606
min,0.000021,0.000020
25%,0.247467,0.253435
50%,0.504131,0.497807
75%,0.758499,0.743126
max,0.999705,0.999692


TF1 test describe


,0,1
count,10000.000000,10000.000000
mean,0.500000,0.500000
std,0.291591,0.291591
min,0.000000,0.000000
25%,0.250000,0.250000
50%,0.500000,0.500000
75%,0.750000,0.750000
max,1.000000,1.000000


TF1 y_train describe


count    5000.000000
mean        0.505381
std         0.166339
min         0.009263
25%         0.414095
50%         0.501150
75%         0.600623
max         0.988818
dtype: float64

TF1 y_test describe


count    10000.000000
mean         0.500000
std          0.170042
min          0.000000
25%          0.404882
50%          0.500000
75%          0.595118
max          1.000000
dtype: float64

In [4]:
# TF1 HKAN model params
tf1_model_params = [
    {
        "layer": 0,
        "n_vars_out": 932,
        "basis_fn": Sigmoid(s=1),
        "n_basis": 2,
        "centers": "random_data_points",
        "expanding_base_regressor": Ridge(alpha=0.1),
    },
    {
        "layer": 1,
        "n_vars_out": 1,
        "basis_fn": Tanh(s=33),
        "n_basis": 13,
        "centers": "random_data_points",
        "expanding_base_regressor": Ridge(alpha=10),
    }
]
tf1_model = utils.build_hkan_model_from_configs(tf1_model_params)
tf1_model

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('expanding_layer_0', ...), ('connecting_layer_0', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,n_vars_out,932
,n_basis,2
,centers,'random_data_points'
,basis_fn,<hkan.hkan.Si...t 0x17c604b90>
,base_regressor,Ridge(alpha=0.1)
,"alpha alpha: {float, ndarray of shape (n_targets,)}, default=1.0Constant that multiplies the L2 term, controlling regularizationstrength. `alpha` must be a non-negative float i.e. in `[0, inf)`.When `alpha = 0`, the objective is equivalent to ordinary leastsquares, solved by the :class:`LinearRegression` object. For numericalreasons, using `alpha = 0` with the `Ridge` object is not advised.Instead, you should use the :class:`LinearRegression` object.If an array is passed, penalties are assumed to be specific to thetargets. Hence they must correspond in number.",0.1
,"fit_intercept fit_intercept: bool, default=TrueWhether to fit the intercept for this model. If setto false, no intercept will be used in calculations(i.e. ``X`` and ``y`` are expected to be centered).",True


In [5]:
# Fit HKAN for TF1
utils.fit_hkan(tf1_model,  tf1_x_train, tf1_y_train)
tf1_results = em.evaluate(
    tf1_model,
    tf1_x_train, tf1_y_train, tf1_x_test, tf1_y_test
)

Fitting connecting regressors: 100%|██████████| 1/1 [00:00<00:00,  2.85it/s]



✓ HKAN training completed in 4.4548 seconds


In [6]:
em.print_results(tf1_results)

Metric          Train                Test                
-------------------------------------------------------
MAE             1.077391e-11         1.226856e-11        
RMSE            1.467517e-11         2.019119e-11        
-------------------------------------------------------
Inference (s)   1.263086             1.807003            


## KAN

In [7]:
tf1_kan_study = utils.study_optuna_kan("TF1_new", tf1_x_train, tf1_y_train, tf1_x_test, tf1_y_test, n_trials = 10)

[I 2026-07-20 15:07:54,736] A new study created in memory with name: no-name-e460c413-83cf-42bb-b6b2-e130f8604128


Optimizing KAN architecture with 5-Fold CV on TF1_new


  0%|          | 0/10 [00:00<?, ?it/s]

checkpoint directory created: ./model
saving model version 0.0


| train_loss: 1.67e-01 | test_loss: 1.62e-01 | reg: 1.79e-01 | : 100%|█| 50/50 [01:08<00:00,  1.37s/


...

| train_loss: 3.43e-01 | test_loss: 3.50e-01 | reg: 4.66e-01 | : 100%|█| 50/50 [00:13<00:00,  3.77it


saving model version 0.7

✓ KAN training with grid extension completed in 199.9212 seconds
checkpoint directory created: ./model
saving model version 0.0


| train_loss: 1.66e-01 | test_loss: 1.69e-01 | reg: 4.57e-01 | :  98%|▉| 49/50 [01:25<00:01,  1.75s/
Best trial: 0. Best value: 0.166323:  10%|█         | 1/10 [40:17<6:02:39, 2417.73s/it]


[W 2026-07-20 15:48:12,476] Trial 1 failed with parameters: {'width_idx': 8} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "/Users/gizemnurdal/miniconda3/envs/swim-meets-kans/lib/python3.11/site-packages/optuna/study/_optimize.py", line 205, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "/Users/gizemnurdal/Workspace/swim-meets-kans/utils.py", line 390, in <lambda>
    lambda trial: objective_kan(
                  ^^^^^^^^^^^^^^
  File "/Users/gizemnurdal/Workspace/swim-meets-kans/utils.py", line 299, in objective_kan
    model = fit_kan_with_grid_extension(model, X_fold_train, y_fold_train,
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/gizemnurdal/Workspace/swim-meets-kans/utils.py", line 222, in fit_kan_with_grid_extension
    model.fit(dataset, opt=opt, steps=steps_per_grid, lamb=lamb)
  File "/Users/gizemnurdal/Workspace/swim-meets-kans/pykan/kan/Mu

KeyboardInterrupt: 

In [ ]:
# Create TF1 KAN model with best architecture
tf1_kan_model = utils.build_kan(width=tf1_kan_study["architecture"], grid=3, k=3, seed=42)

In [ ]:
# Train and test TF1 KAN model
utils.fit_kan(tf1_kan_model, tf1_x_train, tf1_y_train, tf1_x_test, tf1_y_test, steps=100, opt="Adam", lamb=1e-3)

tf1_kan_results = em.evaluate(
    utils.KANModel(tf1_kan_model),
    tf1_x_train, tf1_y_train, tf1_x_test, tf1_y_test
)

In [ ]:
em.print_results(tf1_kan_results)

# TF2 Experiments

## HKAN

In [ ]:
# Load TF2 dataset
tf2 = load_dataset("TF2")
tf2_x_train, tf2_x_test, tf2_y_train, tf2_y_test = tf2["train_input"], tf2["test_input"], tf2["train_label"], tf2["test_label"]

In [ ]:
print("TF2 train describe")
display(pd.DataFrame(tf2_x_train).describe())
print("TF2 test describe")
display(pd.DataFrame(tf2_x_test).describe())
print("TF2 y_train describe")
display(pd.Series(tf2_y_train).describe())
print("TF2 y_test describe")
display(pd.Series(tf2_y_test).describe())

In [ ]:
# TF2 HKAN model params
tf2_model_params = [
    {
        "layer": 0,
        "n_vars_out": 48,
        "basis_fn": Tanh(s=22),
        "n_basis": 17,
        "centers": "random",
        "expanding_base_regressor": Ridge(alpha=0.1),
    },
    {
        "layer": 1,
        "n_vars_out": 11,
        "basis_fn": Softplus(s=18),
        "n_basis": 21,
        "centers": "random_data_points",
        "expanding_base_regressor": Ridge(alpha=1),
    },
    {
        "layer": 2,
        "n_vars_out": 1,
        "basis_fn": Identity(),
        # Default configuration
        "n_basis": 1,
        "centers": "equally_spaced",
        "expanding_base_regressor": None,
    }
]
tf2_model = utils.build_hkan_model_from_configs(tf2_model_params)
tf2_model

In [ ]:
# Fit HKAN for TF2
utils.fit_hkan(tf2_model, tf2_x_train, tf2_y_train)
tf2_results = em.evaluate(
    tf2_model,
    tf2_x_train, tf2_y_train, tf2_x_test, tf2_y_test
)

In [ ]:
em.print_results(tf2_results)

## KAN

In [ ]:
tf2_kan_study = utils.study_optuna_kan("TF2", tf2_x_train, tf2_y_train, tf2_x_test, tf2_y_test, n_trials=10)

In [ ]:
tf2_kan_model = utils.build_kan(width=tf2_kan_study["architecture"], grid=3, k=3, seed=42)

In [ ]:
utils.fit_kan(tf2_kan_model, tf2_x_train, tf2_y_train, tf2_x_test, tf2_y_test, steps=100, opt="Adam", lamb=1e-3)

tf2_kan_results = em.evaluate(
    utils.KANModel(tf2_kan_model),
    tf2_x_train, tf2_y_train, tf2_x_test, tf2_y_test
)

In [ ]:
em.print_results(tf2_kan_results)

# TF3 Experiments

## HKAN

In [ ]:
# Load TF3 dataset
tf3 = load_dataset("TF3")
tf3_x_train, tf3_x_test, tf3_y_train, tf3_y_test = tf3["train_input"], tf3["test_input"], tf3["train_label"], tf3["test_label"]

In [ ]:
print("TF3 train describe")
display(pd.DataFrame(tf3_x_train).describe())
print("TF3 test describe")
display(pd.DataFrame(tf3_x_test).describe())
print("TF3 y_train describe")
display(pd.Series(tf3_y_train).describe())
print("TF3 y_test describe")
display(pd.Series(tf3_y_test).describe())

In [ ]:
# TF3 HKAN model params
tf3_model_params = [
    {
        "layer": 0,
        "n_vars_out": 924,
        "basis_fn": Tanh(s=50),
        "n_basis": 39,
        "centers": "random",
        "expanding_base_regressor": Ridge(alpha=0.001),
    },
    {
        "layer": 1,
        "n_vars_out": 1,
        "basis_fn": Identity(),
        "n_basis": 1,
        "centers": "equally_spaced",
        "expanding_base_regressor": None,
    }
]
tf3_model = utils.build_hkan_model_from_configs(tf3_model_params)
tf3_model

In [ ]:
# Fit HKAN for TF3
utils.fit_hkan(tf3_model, tf3_x_train, tf3_y_train)
tf3_results = em.evaluate(
    tf3_model,
    tf3_x_train, tf3_y_train, tf3_x_test, tf3_y_test
)

In [ ]:
em.print_results(tf3_results)

## KAN

In [ ]:
tf3_kan_study = utils.study_optuna_kan("TF3", tf3_x_train, tf3_y_train, tf3_x_test, tf3_y_test, n_trials=10)

In [ ]:
tf3_kan_model = utils.build_kan(width=tf3_kan_study["architecture"], grid=3, k=3, seed=42)

In [ ]:
utils.fit_kan(tf3_kan_model, tf3_x_train, tf3_y_train, tf3_x_test, tf3_y_test, steps=100, opt="Adam", lamb=1e-3)

tf3_kan_results = em.evaluate(
    utils.KANModel(tf3_kan_model),
    tf3_x_train, tf3_y_train, tf3_x_test, tf3_y_test
)

In [ ]:
em.print_results(tf3_kan_results)

# TF4 Experiments

## HKAN

In [ ]:
# Load TF4 dataset
tf4 = load_dataset("TF4")
tf4_x_train, tf4_x_test, tf4_y_train, tf4_y_test = tf4["train_input"], tf4["test_input"], tf4["train_label"], tf4["test_label"]

In [ ]:
print("TF4 train describe")
display(pd.DataFrame(tf4_x_train).describe())
print("TF4 test describe")
display(pd.DataFrame(tf4_x_test).describe())
print("TF4 y_train describe")
display(pd.Series(tf4_y_train).describe())
print("TF4 y_test describe")
display(pd.Series(tf4_y_test).describe())

In [ ]:
# TF4 HKAN model params
tf4_model_params = [
    {
        "layer": 0,
        "n_vars_out": 1,
        "basis_fn": Tanh(s=3),
        "n_basis": 2,
        "centers": "equally_spaced",
        "expanding_base_regressor": Ridge(alpha=0.001),
    }
]
tf4_model = utils.build_hkan_model_from_configs(tf4_model_params)
tf4_model

In [ ]:
# Fit HKAN for TF4
utils.fit_hkan(tf4_model, tf4_x_train, tf4_y_train)
tf4_results = em.evaluate(
    tf4_model,
    tf4_x_train, tf4_y_train, tf4_x_test, tf4_y_test
)

In [ ]:
em.print_results(tf4_results)

## KAN

In [ ]:
tf4_kan_study = utils.study_optuna_kan("TF4", tf4_x_train, tf4_y_train, tf4_x_test, tf4_y_test, n_trials=10)

In [ ]:
tf4_kan_model = utils.build_kan(width=tf4_kan_study["architecture"], grid=3, k=3, seed=42)

In [ ]:
utils.fit_kan(tf4_kan_model, tf4_x_train, tf4_y_train, tf4_x_test, tf4_y_test, steps=100, opt="Adam", lamb=1e-3)

tf4_kan_results = em.evaluate(
    utils.KANModel(tf4_kan_model),
    tf4_x_train, tf4_y_train, tf4_x_test, tf4_y_test
)

In [ ]:
em.print_results(tf4_kan_results)

# TF5 Experiments

## HKAN

In [ ]:
# Load TF5 dataset
tf5 = load_dataset("TF5")
tf5_x_train, tf5_x_test, tf5_y_train, tf5_y_test = tf5["train_input"], tf5["test_input"], tf5["train_label"], tf5["test_label"]

In [ ]:
print("TF5 train describe")
display(pd.DataFrame(tf5_x_train).describe())
print("TF5 test describe")
display(pd.DataFrame(tf5_x_test).describe())
print("TF5 y_train describe")
display(pd.Series(tf5_y_train).describe())
print("TF5 y_test describe")
display(pd.Series(tf5_y_test).describe())

In [ ]:
# tf5 HKAN model params
tf5_model_params = [
    {
        "layer": 0,
        "n_vars_out": 912,
        "basis_fn": Tanh(s=50),
        "n_basis": 23,
        "centers": "random",
        "expanding_base_regressor": Ridge(alpha=0.01),
    },
    {
        "layer": 1,
        "n_vars_out": 1,
        "basis_fn": Identity(),
        "n_basis": 1,
        "centers": "equally_spaced",
        "equally_spaced": None,
    }
]
tf5_model = utils.build_hkan_model_from_configs(tf5_model_params)
tf5_model

In [ ]:
# Fit HKAN for TF5
utils.fit_hkan(tf5_model, tf5_x_train, tf5_y_train)
tf5_results = em.evaluate(
    tf5_model,
    tf5_x_train, tf5_y_train, tf5_x_test, tf5_y_test
)

In [ ]:
em.print_results(tf5_results)

## KAN

In [ ]:
tf5_kan_study = utils.study_optuna_kan("TF5", tf5_x_train, tf5_y_train, tf5_x_test, tf5_y_test, n_trials=10)

In [ ]:
tf5_kan_model = utils.build_kan(width=tf5_kan_study["architecture"], grid=3, k=3, seed=42)

In [ ]:
utils.fit_kan(tf5_kan_model, tf5_x_train, tf5_y_train, tf5_x_test, tf5_y_test, steps=100, opt="Adam", lamb=1e-3)

tf5_kan_results = em.evaluate(
    utils.KANModel(tf5_kan_model),
    tf5_x_train, tf5_y_train, tf5_x_test, tf5_y_test
)

In [ ]:
em.print_results(tf5_kan_results)